# Additional Baselines: DiffPath

In the following we demonstrate how to reproduce the DiffPath results.

We use the CIFAR-10 --> SVHN and CIFAR-10 --> CelebA setups as examples.

In [ ]:
import io
import pandas as pd
import torch
import numpy as np

from sitn.metrics import bootstrap_auroc
from sitn.utils import construct_results_path, submit_eval_jobs
from sitn.aggregators import GMM

## Run DiffPath Evaluation Jobs

In [ ]:
# Train config
# We assume the model has already been trained with these
# configurations (follow the cross-dataset OOD detection
# notebook to see how).
train_cfg = {"dataset_name": "cifar10"}

# Evaluation configs
eval_cfg_val = {"config": train_cfg, "split_pick": "val", "method": "euler", "step_size": 0.1, "diffpath": True}
eval_cfg_test = {"config": train_cfg, "split_pick": "test", "method": "euler", "step_size": 0.1, "diffpath": True}
eval_cfg_svhn = {"config": train_cfg, "eval_dataset_name": "svhn", "split_pick": "test", "method": "euler", "step_size": 0.1, "diffpath": True}
eval_cfg_celeba = {"config": train_cfg, "eval_dataset_name": "celeba", "split_pick": "test", "method": "euler", "step_size": 0.1, "diffpath": True}

In [ ]:
# Submit slurm jobs for evaluations (or use CLI instead)
# These jobs should only be submitted after training has completed.
submit_eval_jobs([eval_cfg_val, eval_cfg_test, eval_cfg_svhn, eval_cfg_celeba])

Alternatively, via CLI:

`uv run sitn-eval /path/to/training_cfg --split_pick val --method euler --step_size 0.1 --diffpath`

`uv run sitn-eval /path/to/training_cfg --split_pick test --method euler --step_size 0.1 --diffpath`

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name svhn --split_pick test --method euler --step_size 0.1 --diffpath`

`uv run sitn-eval /path/to/training_cfg --eval_dataset_name celeba --split_pick test --method euler --step_size 0.1 --diffpath`

Note that the training config is created and saved in the output folder when a model is trained.

## Fit DiffPath GMM

In [ ]:
DIFFPATH_6D = [
    "dp_score_sum", "dp_score_sum_sq", "dp_score_sum_cb",
    "dp_score_d_dt", "dp_score_d_dt_sq", "dp_score_d_dt_cb"
]

id_val_preds = pd.read_csv(construct_results_path(**eval_cfg_val))

diffpath6d = GMM(features=DIFFPATH_6D)
diffpath6d.fit(id_val_preds)

## Evaluate OOD Detection Performance

In [ ]:
# Metric configurations
metrics = {
    "diffpath6d": {"label": "DiffPath6d", "higher_is_ood": False},
}

# Load ID test predictions
id_preds = pd.read_csv(construct_results_path(**eval_cfg_test, result_type="predictions"))
id_preds["train_dataset"] = eval_cfg_test["config"]["dataset_name"]
id_preds["eval_dataset"] = eval_cfg_test["config"]["dataset_name"]

# Add DiffPath scores
id_preds["diffpath6d"] = diffpath6d.score(id_preds)

results = []
for eval_cfg_ood in [eval_cfg_svhn, eval_cfg_celeba]:
    # Load OOD test predictions
    ood_preds = pd.read_csv(construct_results_path(**eval_cfg_ood, result_type="predictions"))
    ood_preds["train_dataset"] = eval_cfg_ood["config"]["dataset_name"]
    ood_preds["eval_dataset"] = eval_cfg_ood["eval_dataset_name"]

    # Add diffpath scores
    ood_preds["diffpath6d"] = diffpath6d.score(ood_preds)

    # Combine ID and OOD predictions
    preds = pd.concat([id_preds.copy(), ood_preds], ignore_index=True)

    # Compute AUROC with bootstrapped CIs for each method
    y_true = (preds["eval_dataset"] != preds["train_dataset"]).astype(int)
    for col, meta in metrics.items():
        scores = preds[col].copy()
        if not meta["higher_is_ood"]:
            scores = -scores

        auroc, ci_lo, ci_hi = bootstrap_auroc(y_true, scores)
        results.append(
            {
                "ood_dataset": eval_cfg_ood["eval_dataset_name"],
                "metric": meta["label"],
                "AUROC": auroc,
                "CI_lo": ci_lo,
                "CI_hi": ci_hi,
            }
        )

results = pd.DataFrame(results).set_index(["ood_dataset", "metric"])
results

,,AUROC,CI_lo,CI_hi
ood_dataset,metric,,,
svhn,DiffPath6d,0.924733,0.921579,0.927738
celeba,DiffPath6d,0.572744,0.565950,0.579642
